## TechOps Intelligence Platform
### Notebook 02 — Text Pipeline

**Company:** FinTechFlow — B2B Payment Processor  
**Goal:** Embed all FTF text data into ChromaDB

#### Data Embedded
- master_incidents.json  (2000 FTF incidents)
- ftf_postmortems.json   (300 FTF postmortems)
- ftf_playbooks.json     (100 FTF playbooks)

In [11]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import chromadb
PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

PROCESSED  = PROJECT_ROOT / "data/processed"
EMBEDDINGS = PROJECT_ROOT / "data/embeddings"

print("Setup complete")
print(f"Project root: {PROJECT_ROOT}")

Setup complete
Project root: C:\Users\sudha\techops-intelligence


In [12]:
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Model ready — 768 dimensions")

chroma_path = str(EMBEDDINGS / "chroma_db")
client      = chromadb.PersistentClient(path=chroma_path)

collections = {}
for name in ['incidents', 'postmortems', 'playbooks',
             'knowledge_base', 'logs']:
    collections[name] = client.get_or_create_collection(
        name     = name,
        metadata = {"hnsw:space": "cosine"}
    )

print("\nChromaDB collections:")
for name, col in collections.items():
    print(f"  {name:20} : {col.count():,} docs")

Loading embedding model...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Model ready — 768 dimensions

ChromaDB collections:
  incidents            : 2,000 docs
  postmortems          : 175 docs
  playbooks            : 100 docs
  knowledge_base       : 0 docs
  logs                 : 0 docs


In [13]:
BATCH_SIZE = 64

def get_existing_ids(collection) -> set:
    if collection.count() == 0:
        return set()
    return set(collection.get(include=[])['ids'])


def embed_and_store(texts, metadatas, ids,
                    collection, model, batch_size=64):
    stored = 0
    errors = 0

    for i in tqdm(range(0, len(texts), batch_size),
                  desc="Embedding"):
        bt = texts[i:i+batch_size]
        bm = metadatas[i:i+batch_size]
        bi = ids[i:i+batch_size]

        valid = [
            (t, m, id_)
            for t, m, id_ in zip(bt, bm, bi)
            if t and len(t.strip()) > 20
        ]
        if not valid:
            continue

        vt, vm, vi = zip(*valid)
        try:
            emb = model.encode(
                list(vt), show_progress_bar=False
            )
            collection.upsert(
                documents  = list(vt),
                embeddings = emb.tolist(),
                metadatas  = list(vm),
                ids        = list(vi)
            )
            stored += len(vt)
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"Batch error: {e}")

    return stored, errors


print("Helper functions ready")

Helper functions ready


## 1. Embed Master Incidents
2000 FTF incidents covering 8 categories
Used by: triage agent (find similar past incidents)

In [14]:
with open(PROCESSED / "master_incidents.json") as f:
    incidents = json.load(f)

print(f"Loaded: {len(incidents)} FTF incidents")

existing = get_existing_ids(collections['incidents'])
print(f"Already embedded: {len(existing)}")

texts, ids, metadatas = [], [], []

for idx, inc in enumerate(incidents):
    doc_id = f"master_{idx}"
    if doc_id in existing:
        continue

    text = (
        f"{inc.get('title', '')} "
        f"{inc.get('description', '')} "
        f"{inc.get('root_cause', '')}"
    ).strip()

    if len(text) < 20:
        continue

    texts.append(text)
    ids.append(doc_id)
    metadatas.append({
        "source"  : "synthetic_master",
        "severity": inc.get('severity', 'P3'),
        "category": inc.get('category', 'application'),
        "doc_type": "incident"
    })

if texts:
    stored, errors = embed_and_store(
        texts, metadatas, ids,
        collections['incidents'],
        embedding_model, BATCH_SIZE
    )
    print(f"Stored : {stored}")
    print(f"Errors : {errors}")
else:
    print("All incidents already embedded")

print(f"Incidents total: {collections['incidents'].count():,}")

Loaded: 2000 FTF incidents


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Already embedded: 2000
All incidents already embedded
Incidents total: 2,000


## 2. Embed FTF Postmortems
300 FinTechFlow-specific postmortems
Full root cause + resolution + timeline
Used by: diagnosis agent (what caused this before)

In [16]:
with open(PROCESSED / "ftf_postmortems.json") as f:
    postmortems = json.load(f)

print(f"Loaded: {len(postmortems)} FTF postmortems")

existing = get_existing_ids(collections['postmortems'])
texts, ids, metadatas = [], [], []

for idx, pm in enumerate(postmortems):
    doc_id = f"ftf_pm_{idx}"
    if doc_id in existing:
        continue

    res   = pm.get('resolution_steps', [])
    les   = pm.get('lessons_learned', [])
    res_t = ". ".join(res) if isinstance(res, list) else str(res)
    les_t = ". ".join(les) if isinstance(les, list) else str(les)

    text = (
        f"{pm.get('title', '')} "
        f"{pm.get('summary', '')} "
        f"Root cause: {pm.get('root_cause', '')} "
        f"Resolution: {res_t} "
        f"Lessons: {les_t}"
    ).strip()

    if len(text) < 30:
        continue

    texts.append(text)
    ids.append(doc_id)
    metadatas.append({
        "source"  : "ftf_postmortem",
        "severity": pm.get('severity', 'P2'),
        "category": pm.get('category', 'application'),
        "team"    : pm.get('team', 'platform-sre'),
        "doc_type": "postmortem"
    })

if texts:
    stored, errors = embed_and_store(
        texts, metadatas, ids,
        collections['postmortems'],
        embedding_model, BATCH_SIZE
    )
    print(f"Stored : {stored}")
    print(f"Errors : {errors}")
else:
    print("All postmortems already embedded")

print(f"Postmortems total: {collections['postmortems'].count():,}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded: 175 FTF postmortems
All postmortems already embedded
Postmortems total: 175


## 3. Embed FTF Playbooks
100 FinTechFlow incident response playbooks
Phase-by-phase response procedures
Used by: resolution agent (how to respond)

In [20]:
with open(PROCESSED / "ftf_playbooks.json") as f:
    playbooks = json.load(f)

print(f"Loaded: {len(playbooks)} FTF playbooks")

existing = get_existing_ids(collections['playbooks'])
texts, ids, metadatas = [], [], []

for idx, pb in enumerate(playbooks):
    doc_id = f"ftf_pb_{idx}"
    if doc_id in existing:
        continue

    phases     = pb.get('response_phases', [])
    phase_text = ""
    if isinstance(phases, list):
        for phase in phases:
            if isinstance(phase, dict):
                actions = phase.get('actions', [])
                if isinstance(actions, list):
                    phase_text += (
                        f"{phase.get('phase', '')}: "
                        f"{'. '.join(actions)} "
                    )

    triage   = pb.get('triage_steps', [])
    triage_t = ". ".join(triage) \
        if isinstance(triage, list) else ""

    text = (
        f"{pb.get('title', '')} "
        f"Trigger: {pb.get('trigger', '')} "
        f"Triage: {triage_t} "
        f"{phase_text}"
        f"Escalation: {pb.get('escalation_criteria', '')}"
    ).strip()

    if len(text) < 30:
        continue

    texts.append(text)
    ids.append(doc_id)
    metadatas.append({
        "source"  : "ftf_playbook",
        "severity": pb.get('severity', 'P2'),
        "category": pb.get('category', 'application'),
        "team"    : pb.get('team', 'platform-sre'),
        "doc_type": "playbook"
    })

if texts:
    stored, errors = embed_and_store(
        texts, metadatas, ids,
        collections['playbooks'],
        embedding_model, BATCH_SIZE
    )
    print(f"Stored : {stored}")
    print(f"Errors : {errors}")
else:
    print("All playbooks already embedded")

print(f"Playbooks total: {collections['playbooks'].count():,}")

Loaded: 100 FTF playbooks
All playbooks already embedded
Playbooks total: 100


## 4. Retrieval Quality Test
Verify embeddings return relevant FTF content

In [21]:
def search(query, collection_name, n=2):
    emb = embedding_model.encode([query]).tolist()
    r   = collections[collection_name].query(
        query_embeddings = emb,
        n_results        = n,
        include          = ['documents', 'metadatas', 'distances']
    )
    return r


test_queries = [
    ("PostgreSQL connection refused port 5432",
     "incidents",   "database incident"),
    ("payment service OOMKilled memory limit",
     "incidents",   "memory incident"),
    ("Vault sealed during KMS key rotation",
     "postmortems", "security postmortem"),
    ("ALB all targets unhealthy response procedure",
     "playbooks",   "network playbook"),
    ("Kafka consumer lag critical response",
     "playbooks",   "kafka playbook"),
]

print("Retrieval quality test\n")
for query, collection, expected in test_queries:
    r     = search(query, collection, n=1)
    score = 1 - r['distances'][0][0]
    src   = r['metadatas'][0][0].get('source', 'unknown')
    text  = r['documents'][0][0][:100]
    print(f"Query    : {query}")
    print(f"Expected : {expected}")
    print(f"Score    : {score:.3f}  Source: {src}")
    print(f"Result   : {text}")
    print()

Retrieval quality test

Query    : PostgreSQL connection refused port 5432
Expected : database incident
Score    : 0.753  Source: synthetic_master
Result   : PostgreSQL Primary Connection Refusal A connection refused error (ECONNREFUSED on port 5432) occurre

Query    : payment service OOMKilled memory limit
Expected : memory incident
Score    : 0.831  Source: synthetic_master
Result   : Payment Service JVM OOMKilled The Payment service JVM experienced an OOMKilled error, halting all pa

Query    : Vault sealed during KMS key rotation
Expected : security postmortem
Score    : 0.714  Source: ftf_postmortem
Result   : HashiCorp Vault sealed during AWS KMS key rotation window HashiCorp Vault sealed during a KMS key ro

Query    : ALB all targets unhealthy response procedure
Expected : network playbook
Score    : 0.603  Source: ftf_playbook
Result   : ALB target group health check failure Trigger: PagerDuty alert fired Triage: Assess scope. Check das

Query    : Kafka consumer lag critical

In [23]:
print("=" * 50)
print("NOTEBOOK 02 - TEXT PIPELINE COMPLETE")
print("=" * 50)
for name, col in collections.items():
    print(f"  {name:20} : {col.count():,} docs")
print("\nNext: Notebook 03 - PDF Pipeline")
print("=" * 50)

NOTEBOOK 02 - TEXT PIPELINE COMPLETE
  incidents            : 2,000 docs
  postmortems          : 175 docs
  playbooks            : 100 docs
  knowledge_base       : 0 docs
  logs                 : 0 docs

Next: Notebook 03 - PDF Pipeline
